In [94]:
import scipy
import numpy as np
import pandas as pd

from datasets import load_dataset

from sklearn.utils import shuffle
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import euclidean_distances

import os
from pprint import pprint
from time import perf_counter as pf
from tqdm import tqdm

import torch
from torch import nn
from torch.nn import functional as F

In [2]:
ds = load_dataset('tattabio/ec_classification')
train = ds['train']
test = ds['test']

In [3]:
train_entries = train.to_pandas()['Entry']
test_entries  = test.to_pandas()['Entry']

In [ ]:
train

In [14]:
len(train_entries.loc[train_entries == 'A0AFT'].index)

0

In [17]:
def find_file(dir, uniprot_id):
    filenames = os.listdir(dir)
    for fname in filenames:
        if uniprot_id in fname:
            return pd.read_csv(os.path.join(dir, fname))
    
    return None

find_file('data/ec_classification/secondary_centroids', 'Q59478')

,id,conf_type_id,beg_chain,beg_seq,beg_res,end_chain,end_seq,end_res,n_atoms,mean_x,mean_y,mean_z
0,HELX_LH_PP_P1,HELX_LH_PP_P,A,20,ALA,A,23,PRO,24,9.164,-17.119,11.241
1,HELX_RH_3T_P1,HELX_RH_3T_P,A,25,PRO,A,27,ASP,19,5.554,-13.986,2.829
2,TURN_TY1_P1,TURN_TY1_P,A,28,LYS,A,28,LYS,9,0.880,-17.277,1.432
3,BEND1,BEND,A,29,PHE,A,29,PHE,11,0.575,-13.000,-0.315
4,TURN_TY1_P2,TURN_TY1_P,A,32,SER,A,33,GLY,10,6.820,-8.430,-7.573
...,...,...,...,...,...,...,...,...,...,...,...,...
73,BEND24,BEND,A,284,LYS,A,286,GLY,20,-4.099,15.037,12.145
74,TURN_TY1_P23,TURN_TY1_P,A,288,PRO,A,289,ASP,15,-0.282,8.212,18.534
75,STRN22,STRN,A,291,TYR,A,302,THR,103,-5.322,-6.414,1.897
76,BEND25,BEND,A,304,GLY,A,304,GLY,4,-4.763,-6.814,-20.690


In [18]:
train_secondary_centroids = {}
test_secondary_centroids = {}
for row in tqdm(train):
    maybe_csv = find_file('data//ec_classification//secondary_centroids//', row['Entry'])
    
    if maybe_csv is not None:
        train_secondary_centroids[row['Entry']] = maybe_csv

for row in tqdm(test):
    maybe_csv = find_file('data//ec_classification//secondary_centroids//', row['Entry'])
    
    if maybe_csv is not None:
        test_secondary_centroids[row['Entry']] = maybe_csv

100%|██████████| 128/128 [00:00<00:00, 904.18it/s]


In [75]:
train_labels = []
for entry in train_secondary_centroids.keys():
    idxrange = train_entries.loc[train_entries == entry].index
    if len(idxrange) > 0:
        train_labels.append(train[idxrange[0]]['Label'])

test_labels = []
for entry in test_secondary_centroids.keys():
    idxrange = test_entries.loc[test_entries == entry].index
    if len(idxrange) > 0:
        test_labels.append(test[idxrange[0]]['Label'])

In [139]:
# find the label encoding
serieses = []
for key in train_secondary_centroids.keys():
    serieses.append(train_secondary_centroids[key].conf_type_id)

for key in test_secondary_centroids.keys():
    s = train_secondary_centroids.get(key)
    if s is not None:
        serieses.append(s)

labenc = LabelEncoder()
print(pd.concat(serieses).unique())
ids = labenc.fit_transform(pd.concat(serieses).unique())
print(ids)

<ArrowStringArray>
['HELX_LH_PP_P',   'TURN_TY1_P', 'HELX_RH_3T_P', 'HELX_RH_AL_P',
         'BEND', 'HELX_RH_PI_P',         'STRN']
Length: 7, dtype: str
[1 6 2 3 0 4 5]


In [142]:
def df_to_map(df):
    distances = euclidean_distances(df[['mean_x', 'mean_y', 'mean_z']])
    encoded_conf_type = labenc.transform(df.conf_type_id)
    c1 = np.repeat(np.array(encoded_conf_type)[:, np.newaxis], repeats=len(encoded_conf_type), axis=1)
    c2 = c1.T
    c1 = F.one_hot(torch.tensor(c1), num_classes=len(ids))
    c2 = F.one_hot(torch.tensor(c2), num_classes=len(ids))

    return torch.concat([torch.unsqueeze(torch.tensor(distances), -1), c1, c2], dim = -1).permute(2, 0, 1).to(torch.float32)


In [143]:
train_structures = []
for key in train_secondary_centroids.keys():
    train_structures.append(df_to_map(train_secondary_centroids[key]))

test_structures = []
for key in test_secondary_centroids.keys():
    test_structures.append(df_to_map(test_secondary_centroids[key]))

In [144]:
y_train = np.array(list(map(lambda x: int(x[0])-1, train_labels)))
y_test = np.array(list(map(lambda x: int(x[0])-1, test_labels)))

In [145]:
train_structures, y_train = shuffle(train_structures, y_train)
test_structures, y_test = shuffle(test_structures, y_test)

In [146]:
np.unique(y_test)

array([0, 1, 2, 3, 4, 5, 6])

In [201]:
model = nn.Sequential(
    nn.Conv2d(15, 64, 3, padding=1),
    nn.ReLU(),
    # nn.GroupNorm(num_groups=1, num_channels=64),

    nn.Conv2d(64, 64, 3, padding=1),
    nn.ReLU(),
    # nn.GroupNorm(num_groups=1, num_channels=64),

    nn.Conv2d(64, 64, 3, padding=1),
    nn.ReLU(),
    # nn.GroupNorm(num_groups=1, num_channels=64),

    nn.Conv2d(64, 64, 3, padding=1),
    nn.ReLU(),
    # nn.GroupNorm(num_groups=1, num_channels=64),

    nn.Conv2d(64, 64, 3, padding=1),
    nn.ReLU(),
    # nn.GroupNorm(num_groups=1, num_channels=64),

    nn.Conv2d(64, 64, 3, padding=1),
    nn.ReLU(),
    # nn.GroupNorm(num_groups=1, num_channels=64),


    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(0),
    nn.Linear(64, 7)
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [202]:
y_pred = model(train_structures[0])

In [203]:
accumulation_steps = 16

for epoch in range(10):
    for i, (X, y) in enumerate(zip(train_structures, y_train)):
        y_pred = model(X)
        loss = criterion(y_pred, torch.tensor(y).to(torch.long))
        loss = loss / accumulation_steps
        
        # Backward pass - gradients accumulate in .grad attributes
        loss.backward()
        
        # Only update weights every N steps
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()        # Update weights
            optimizer.zero_grad()


In [204]:
model.eval()
total_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for X, y in zip(test_structures, y_test):
        X = X.unsqueeze(0)
        y = torch.tensor(y).to(torch.long).unsqueeze(0)
        # print(y.shape)

        y_pred = model(X)
        loss = criterion(y_pred, y)
        total_loss += loss.item()

        # print(torch.max(y_pred))
        predicted = torch.argmax(y_pred)  # Shape: (1,)
        correct += (predicted == y).sum().item()
        total += 1
    
    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total
    
print('avg_loss:', avg_loss)
print('accuracy:', accuracy)

avg_loss: 1.5105288998555328
accuracy: 44.067796610169495


In [176]:
torch.argmax(y_pred)

tensor(2)